# 02_3 DQA Control Sweep 6h

02_2では、client pseudo-label trainingそのものがwarmupを壊しやすく、DQA/FedAvgの差はかなり小さいことが見えました。

このNotebookでは、追加のclient学習を増やさず、02_2で作ったclient checkpointsとpseudo statsを再利用して、DQAをどう制御すれば機能するのかを検証します。

主な検証:

- DQA hyperparameter sweep: server floor, residual blend, classwise blend, temperature, count/quality gate
- client subset ablation: overcast/rainy/snowy単独、leave-one-client-out、全client
- FedAvgとの差分: 同じclient subsetでDQAがFedAvgを超える条件
- warmup rollback rule: warmup未満なら採用しない判断表
- top候補だけserver calibrationを1 epoch実行

想定実行時間は、02_2のclient checkpointsが存在している状態で6時間以内です。出力は `dynamic_quality_aware_classwise_aggregation/exploring/runs/02_3_dqa_control_sweep_6h/` に保存します。

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "Object_Detection" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "Object_Detection":
    raise RuntimeError("Run this notebook from inside /app/Object_Detection")

EXPLORING_ROOT = ROOT / "dynamic_quality_aware_classwise_aggregation" / "exploring"
if str(EXPLORING_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPLORING_ROOT))

from dqa_probe_02_3 import ControlSweepConfig, DQAControlSweep

cfg = ControlSweepConfig(
    source_experiment_name="02_2_dqa_functionality_probe_7h",
    experiment_name="02_3_dqa_control_sweep_6h",
    source_warmup_key="dqa_v2_scene_12h",
    max_wall_hours=6.0,
    server_train_limit=2048,
    server_val_limit=1024,
    client_target_limit=2048,
    device_cli="0",
    batch_size=8,
    val_batch_size=1,
    workers=0,
    force_rerun=False,
    run_top_calibration=True,
    top_calibration_k=8,
)

sweep = DQAControlSweep(cfg)
sweep.describe()

## 2. Source Check

02_2のclient checkpointsとpseudo statsが揃っているか確認します。ここがmissingなら、先に02_2のclient trainingまで回してください。

In [ ]:
source_status = sweep.source_status()
source_status

In [ ]:
pseudo = sweep.pseudo_summary()
cols = ["variant", "client_id", "weather", "pseudo_total", "active_classes", "zero_classes", "top_class_share", "mean_quality_active"]
cols = [c for c in cols if c in pseudo.columns]
pseudo[cols].sort_values(["variant", "client_id"])

## 3. Run Control Sweep

全client / 単独client / leave-one-outに対して、FedAvgと複数のDQA制御設定を評価します。途中で止まっても既存JSON/checkpointはskipします。

In [ ]:
leaderboard = sweep.run_all()
display_cols = [
    "name", "kind", "variant", "subset", "candidate", "status",
    "map50_final", "map50_95_final", "delta_map50_95_vs_warmup", "delta_vs_fedavg",
    "precision_final", "recall_final", "source_table",
]
display_cols = [c for c in display_cols if c in leaderboard.columns]
leaderboard[display_cols].head(40)

## 4. Decision Table

`decision_table.csv` は、DQAを採用すべきかの判断表です。

- `delta_map50_95_vs_warmup > 0`: warmupより良いので採用候補
- `delta_vs_fedavg > 0`: 同じclient subsetでFedAvgよりDQAが良い
- `subset`: どのweather clientが効いた/壊したかを見るための軸
- `candidate`: DQA制御設定

In [ ]:
decision = sweep.build_decision_table()
decision_cols = [
    "variant", "subset", "candidate", "status", "map50_final", "map50_95_final",
    "delta_map50_95_vs_warmup", "delta_vs_fedavg", "accept_over_warmup", "accept_over_fedavg",
    "precision_final", "recall_final", "active_classes", "candidate_note",
]
decision_cols = [c for c in decision_cols if c in decision.columns]
decision[decision_cols].head(60)

## 5. Subset And Candidate Views

DQAが効く条件を見るため、variant/subset/candidateごとのbestをざっくり集計します。

In [ ]:
import pandas as pd

if not decision.empty:
    best_by_subset = (
        decision.sort_values("map50_95_final", ascending=False)
        .groupby(["variant", "subset"], as_index=False)
        .first()
    )
    subset_cols = ["variant", "subset", "candidate", "map50_final", "map50_95_final", "delta_map50_95_vs_warmup", "delta_vs_fedavg"]
    display(best_by_subset[[c for c in subset_cols if c in best_by_subset.columns]].sort_values("map50_95_final", ascending=False).head(40))
else:
    print("decision table is empty")

In [ ]:
if not decision.empty:
    candidate_view = (
        decision.groupby("candidate", as_index=False)
        .agg(
            mean_map50_95=("map50_95_final", "mean"),
            best_map50_95=("map50_95_final", "max"),
            mean_delta_vs_fedavg=("delta_vs_fedavg", "mean"),
            accepted_over_warmup=("accept_over_warmup", "sum"),
        )
        .sort_values("best_map50_95", ascending=False)
    )
    candidate_view
else:
    print("decision table is empty")

## 6. Top Calibration

`run_all()` でtop候補だけserver calibration済みです。必要ならこのセルで再表示します。DQAがaggregation直後に弱くても、calibration後にwarmupを超えるなら、DQA後のserver repairを必須にする価値があります。

In [ ]:
cal_path = sweep.result_root / "top_calibration_summary.csv"
if cal_path.exists():
    cal = pd.read_csv(cal_path)
    cal_cols = ["source_control_name", "variant", "subset", "candidate", "status", "map50_final", "map50_95_final", "precision_final", "recall_final"]
    cal[[c for c in cal_cols if c in cal.columns]].sort_values("map50_95_final", ascending=False)
else:
    print("No top calibration summary yet")

## 7. Interpretation

見るべき結論:

- DQAがFedAvgより勝つcandidateがあるか
- 全clientより単独/leave-one-outが良いなら、特定weather clientが更新を壊している
- `floor80/floor90` が強いなら、client residualはかなり小さくするべき
- `count_gate` / `quality_gate` が強いなら、pseudo statsによるclass gatingが必要
- top calibration後だけ良いなら、DQA aggregation単体ではなくDQA + server repairを1セットにする
- warmup未満しか出ないなら、phase2 client updateはrollback対象にする